In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

# Pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [10]:
# Load the dataset
dataset_path = './b3427ed8ad063a09_MOHANAD_A4706/data/NF-CSE-CIC-IDS2018-v2.csv'
df = pd.read_csv(dataset_path, nrows=5000000)

print(f"Columns: {df.columns.tolist()}")
print("\nFirst 5 rows:")
df.head(3)

Columns: ['IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'Label', 'Attack']

First 5 rows:


,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,DURATION_IN,DURATION_OUT,MIN_TTL,MAX_TTL,LONGEST_FLOW_PKT,SHORTEST_FLOW_PKT,MIN_IP_PKT_LEN,MAX_IP_PKT_LEN,SRC_TO_DST_SECOND_BYTES,DST_TO_SRC_SECOND_BYTES,RETRANSMITTED_IN_BYTES,RETRANSMITTED_IN_PKTS,RETRANSMITTED_OUT_BYTES,RETRANSMITTED_OUT_PKTS,SRC_TO_DST_AVG_THROUGHPUT,DST_TO_SRC_AVG_THROUGHPUT,NUM_PKTS_UP_TO_128_BYTES,NUM_PKTS_128_TO_256_BYTES,NUM_PKTS_256_TO_512_BYTES,NUM_PKTS_512_TO_1024_BYTES,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label,Attack
0,13.58.98.64,40894,172.31.69.25,22,6,92.0,3164,23,3765,21,27,27,27,0,0,0,63,63,1028,52,52,1028,3164.0,3765.0,0,0,0,0,25312000,30120000,33,7,1,2,1,26883,26847,0,0,0,0,0,0,1,SSH-Bruteforce
1,213.202.230.143,29622,172.31.66.103,3389,6,0.0,1919,14,2031,11,223,219,30,0,0,0,101,101,1195,40,40,1195,1919.0,2031.0,0,0,0,0,15352000,16248000,17,6,0,1,1,8192,64000,0,0,0,0,0,0,0,Benign
2,172.31.66.5,65456,172.31.0.2,53,17,0.0,116,2,148,2,0,0,0,0,0,0,128,128,74,58,58,74,116.0,148.0,0,0,0,0,928000,1184000,4,0,0,0,0,0,0,0,0,2511,1,5,0,0,Benign
3,172.31.64.92,57918,172.31.0.2,53,17,0.0,70,1,130,1,0,0,0,0,0,0,0,0,130,70,70,130,70.0,130.0,0,0,0,0,560000,1040000,1,1,0,0,0,0,0,0,0,3371,1,60,0,0,Benign
4,18.219.32.43,63269,172.31.69.25,80,6,7.0,232,5,1136,4,223,222,27,4294827,140,0,127,127,1004,40,40,1004,232.0,1136.0,0,0,0,0,8000,9088000,8,0,0,1,0,8192,26883,0,0,0,0,0,0,1,DDoS attacks-LOIC-HTTP


In [ ]:
cols = df.columns.tolist()
print([col for col in cols if 'time' in col.lower() or 'start' in col.lower() or 'first' in col.lower()])

[]


In [12]:
df['start_time'] = 0
for i in range(1, len(df)):
    df.loc[i, 'start_time'] = df.loc[i-1, 'start_time'] + df.loc[i-1, 'FLOW_DURATION_MILLISECONDS']

# 2. Create 2‑minute window index
df['window'] = (df['start_time'] // 120_000).astype(int)   # integer window number

# 3. Define internal IP addresses (example: keep everything for now, or filter private ranges)
# The dataset uses public and private IPs; we'll keep all for simplicity.
# To mimic the paper, you could filter to 172.31.0.0/16 (common private range in this dataset)
internal_ips = set(df['IPV4_SRC_ADDR'].unique()) | set(df['IPV4_DST_ADDR'].unique())
# But that's too big; instead we'll just work with all IPs.

# 4. Aggregate: source‑based and destination‑based
# Source aggregates: group by source IP + window
src_agg = df.groupby(['IPV4_SRC_ADDR', 'window']).agg(
    n_fwd_pkts=('IN_PKTS', 'sum'),
    n_bwd_pkts=('OUT_PKTS', 'sum'),
    sum_flx_dur=('FLOW_DURATION_MILLISECONDS', 'sum'),
    tot_flx=('FLOW_DURATION_MILLISECONDS', 'count'),
    sum_pkts_size=('IN_BYTES', 'sum') + ('OUT_BYTES', 'sum'),  # total bytes
    std_pkt_size=('MAX_IP_PKT_LEN', 'std'),   # approximation
    n_dst_ports=('L4_DST_PORT', 'nunique'),
    n_dst_ip=('IPV4_DST_ADDR', 'nunique')    # for source perspective
).reset_index()

# Destination aggregates: group by destination IP + window
dst_agg = df.groupby(['IPV4_DST_ADDR', 'window']).agg(
    n_fwd_pkts=('IN_PKTS', 'sum'),
    n_bwd_pkts=('OUT_PKTS', 'sum'),
    sum_flx_dur=('FLOW_DURATION_MILLISECONDS', 'sum'),
    tot_flx=('FLOW_DURATION_MILLISECONDS', 'count'),
    sum_pkts_size=('IN_BYTES', 'sum') + ('OUT_BYTES', 'sum'),
    std_pkt_size=('MAX_IP_PKT_LEN', 'std'),
    n_src_ports=('L4_SRC_PORT', 'nunique'),
    n_src_ip=('IPV4_SRC_ADDR', 'nunique')    # for destination perspective
).reset_index()

# 5. Label each aggregate as "attack" if any flow inside it has Label == 1
# Merge with per‑flow labels? Simpler: compute from original df
attack_src = df.groupby(['IPV4_SRC_ADDR', 'window'])['Label'].max().reset_index()
attack_src.columns = ['IPV4_SRC_ADDR', 'window', 'is_attack']
src_agg = src_agg.merge(attack_src, on=['IPV4_SRC_ADDR', 'window'], how='left')
src_agg['is_attack'] = src_agg['is_attack'].fillna(0).astype(int)

attack_dst = df.groupby(['IPV4_DST_ADDR', 'window'])['Label'].max().reset_index()
attack_dst.columns = ['IPV4_DST_ADDR', 'window', 'is_attack']
dst_agg = dst_agg.merge(attack_dst, on=['IPV4_DST_ADDR', 'window'], how='left')
dst_agg['is_attack'] = dst_agg['is_attack'].fillna(0).astype(int)

TypeError: Must provide 'func' or tuples of '(column, aggfunc).

## 2. Data Types and Missing Values Analysis

In [ ]:
# Data types summary
print("Data Types:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Missing values analysis
missing_values = df.isnull().sum()
missing_pct = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing_Count': missing_values.values,
    'Missing_%': missing_pct.values
}).sort_values('Missing_Count', ascending=False)

print("Missing Values Summary:")
print(missing_df[missing_df['Missing_Count'] > 0].to_string(index=False) if (missing_df['Missing_Count'] > 0).any() else "No missing values found!")

In [ ]:
# Visualize missing values if any exist
missing_with_values = missing_df[missing_df['Missing_Count'] > 0]
if len(missing_with_values) > 0:
    fig = px.bar(missing_with_values.sort_values('Missing_Count', ascending=True),
                 x='Missing_Count', y='Column',
                 orientation='h',
                 title='Missing Values by Column',
                 labels={'Missing_Count': 'Count', 'Column': 'Columns'},
                 color='Missing_%',
                 color_continuous_scale='Reds')
    fig.show()
else:
    print("No missing values to visualize - dataset is complete!")

## 3. Statistical Summary of Features

In [ ]:
# Get numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Number of numeric features: {len(numeric_cols)}\n")

# Display statistical summary
print("Statistical Summary of Numeric Features:")
df[numeric_cols].describe()

## 4. Distribution Analysis of Key Features

In [ ]:
# Select top features by variance for distribution analysis
feature_variance = df[numeric_cols].var().sort_values(ascending=False)
top_features = feature_variance.head(12).index.tolist()

print("Top 12 Features by Variance:")
print(feature_variance.head(12))

In [ ]:
# Distribution plots for top features
n_features = len(top_features)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 12))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    axes[idx].hist(df[feature].dropna(), bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    axes[idx].set_title(f'{feature}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

# Hide extra subplots
for idx in range(n_features, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation_matrix = df[numeric_cols].corr()

# Find highly correlated feature pairs
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.95:
            high_corr_pairs.append({
                'Feature_1': correlation_matrix.columns[i],
                'Feature_2': correlation_matrix.columns[j],
                'Correlation': correlation_matrix.iloc[i, j]
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
    print(f"Highly Correlated Feature Pairs (|r| > 0.95): {len(high_corr_df)}")
    print(high_corr_df.head(10).to_string(index=False))
else:
    print("No highly correlated feature pairs found (|r| > 0.95)")

In [ ]:
# Correlation heatmap for top features
top_corr_features = feature_variance.head(15).index.tolist()
corr_subset = df[top_corr_features].corr()

fig = px.imshow(corr_subset,
                labels=dict(color='Correlation'),
                title='Correlation Matrix: Top 15 Features by Variance',
                color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1,
                height=700, width=700)
fig.show()

## 6. Class Distribution Analysis

In [ ]:
# Identify categorical columns (potential class labels)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns: {categorical_cols}\n")

# Check the last column as it's often the label column
if len(df.columns) > 0:
    last_col = df.columns[-1]
    print(f"Last column: {last_col}")
    print(f"Unique values: {df[last_col].nunique()}")
    print(f"\nValue counts:")
    print(df[last_col].value_counts())

In [ ]:
# Visualize class distribution for the last column (assumed to be label)
if len(df.columns) > 0:
    last_col = df.columns[-1]
    class_counts = df[last_col].value_counts()
    class_pct = (class_counts / len(df)) * 100
    
    # Create visualization
    fig = px.bar(x=class_counts.index, y=class_counts.values,
                 title=f'Class Distribution: {last_col}',
                 labels={'x': 'Class', 'y': 'Count'},
                 color=class_counts.values,
                 color_continuous_scale='Viridis',
                 text='value')
    fig.update_traces(textposition='outside')
    fig.show()
    
    # Print summary
    print("\nClass Distribution Summary:")
    dist_summary = pd.DataFrame({
        'Class': class_counts.index,
        'Count': class_counts.values,
        'Percentage': class_pct.values
    })
    print(dist_summary.to_string(index=False))

## 7. Feature Relationships and Interactions

In [ ]:
# Create scatter plots for top feature pairs
if len(top_features) >= 2:
    # Get top 4 feature pairs
    top_4_features = top_features[:4]
    
    fig = make_subplots(rows=2, cols=2, 
                        subplot_titles=[(f"{top_4_features[i]} vs {top_4_features[(i+1) % len(top_4_features)]}") for i in range(4)])
    
    for idx in range(4):
        feat1 = top_4_features[idx % len(top_4_features)]
        feat2 = top_4_features[(idx + 1) % len(top_4_features)]
        
        row = idx // 2 + 1
        col = idx % 2 + 1
        
        # Sample data for better visualization if dataset is large
        sample_size = min(1000, len(df))
        sample_indices = np.random.choice(len(df), sample_size, replace=False)
        
        fig.add_trace(
            go.Scatter(x=df.iloc[sample_indices][feat1],
                      y=df.iloc[sample_indices][feat2],
                      mode='markers',
                      marker=dict(size=3, opacity=0.5),
                      name=f'{feat1} vs {feat2}'),
            row=row, col=col
        )
    
    fig.update_layout(height=800, title_text="Feature Relationships (Top Feature Pairs)")
    fig.show()

In [ ]:
# Box plots of top features by class
if len(df.columns) > 0 and len(top_features) >= 4:
    last_col = df.columns[-1]
    
    fig = make_subplots(rows=2, cols=2, subplot_titles=top_features[:4])
    
    for idx, feature in enumerate(top_features[:4]):
        row = idx // 2 + 1
        col = idx % 2 + 1
        
        for class_val in df[last_col].unique()[:10]:  # Limit to first 10 classes
            class_data = df[df[last_col] == class_val][feature].dropna()
            fig.add_trace(
                go.Box(y=class_data, name=str(class_val), boxmean='sd'),
                row=row, col=col
            )
    
    fig.update_layout(height=800, title_text=f"Feature Distributions by {last_col} (Top 4 Features)")
    fig.show()

## 8. Anomaly Detection Overview

In [ ]:
# Outlier detection using IQR method
def find_outliers_iqr(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return ((data < lower_bound) | (data > upper_bound)).sum()

outlier_summary = {}
for feature in top_features:
    outlier_count = find_outliers_iqr(df[feature].dropna())
    outlier_pct = (outlier_count / len(df)) * 100
    outlier_summary[feature] = {'count': outlier_count, 'percentage': outlier_pct}

outlier_df = pd.DataFrame(outlier_summary).T.sort_values('count', ascending=False)
print("Outliers by Feature (IQR method):")
print(outlier_df.head(10))

In [ ]:
# Visualize outliers
outlier_data = outlier_df.reset_index().rename(columns={'index': 'Feature'}).head(12)
fig = px.bar(outlier_data.sort_values('count', ascending=True),
             x='count', y='Feature',
             orientation='h',
             title='Outlier Count by Feature (IQR Method)',
             labels={'count': 'Outlier Count', 'Feature': 'Features'},
             color='count',
             color_continuous_scale='Reds')
fig.show()

## 9. Data Quality and Summary

In [ ]:
# Calculate data quality metrics
numeric_data = df.select_dtypes(include=[np.number])

# Completeness
completeness = (1 - numeric_data.isnull().sum().sum() / (len(numeric_data) * len(numeric_data.columns))) * 100

# Duplicates
duplicates = df.duplicated().sum()
duplicate_pct = (duplicates / len(df)) * 100

# Infinite values
inf_count = np.isinf(numeric_data).sum().sum()

print("DATA QUALITY METRICS")
print("=" * 60)
print(f"Total Records: {len(df):,}")
print(f"Total Features: {len(df.columns)}")
print(f"Numeric Features: {len(numeric_cols)}")
print(f"Categorical Features: {len(categorical_cols)}")
print(f"\nCompleteness: {completeness:.2f}%")
print(f"Duplicate Records: {duplicates:,} ({duplicate_pct:.4f}%)")
print(f"Infinite Values: {inf_count}")
print(f"Total Outliers Detected: {int(outlier_df['count'].sum()):,}")
print("=" * 60)

## 10. Key Insights and Recommendations

In [ ]:
print("KEY FINDINGS & RECOMMENDATIONS")
print("=" * 70)

print("\n1. DATASET CHARACTERISTICS:")
print(f"   • Total samples: {len(df):,}")
print(f"   • Feature dimensionality: {len(numeric_cols)} numeric features")
print(f"   • Data completeness: {completeness:.2f}%")

print(f"\n2. FEATURE STATISTICS:")
print(f"   • Highest variance feature: {feature_variance.idxmax()}")
print(f"   • Variance range: {feature_variance.min():.2e} to {feature_variance.max():.2e}")
print(f"   • Features with outliers: {len(outlier_df[outlier_df['count'] > 0])}")

if len(high_corr_pairs) > 0:
    print(f"\n3. FEATURE RELATIONSHIPS:")
    print(f"   • Highly correlated pairs detected: {len(high_corr_pairs)}")
    print(f"   • Recommendation: Consider feature selection to reduce multicollinearity")
else:
    print(f"\n3. FEATURE RELATIONSHIPS:")
    print(f"   • Low multicollinearity detected (no |r| > 0.95)")

print(f"\n4. DATA QUALITY:")
print(f"   • Duplicate records: {duplicates:,}")
print(f"   • Missing values: {numeric_data.isnull().sum().sum()}")
print(f"   • Recommendation: Data appears clean and ready for preprocessing")

print(f"\n5. RECOMMENDATIONS FOR PREPROCESSING:")
print(f"   • Apply outlier handling (IQR or robust scaling)")
print(f"   • Consider feature normalization/standardization")
print(f"   • Evaluate feature importance for feature selection")
print(f"   • Check class balance and apply appropriate techniques if needed")

print("=" * 70)